In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("./Datasets/initial_cleaned.csv", index_col=0)
df = df.reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2072, 44)


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d,reference
0,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,132.0,Ref-1-data
1,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,122.5,Ref-1-data
2,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,116.0,Ref-1-data
3,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,134.0,Ref-1-data
4,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,131.5,Ref-1-data


# Semantic Missingness Analysis

Analyzing and recoding missing values in UHPC dataset, distinguishing between structural zeros (not applicable) and actual missing data (unknown).

In [2]:
## Data Filtering Utility

def drop_by_threshold(df, threshold_pct):
    """Drop columns where missing values exceed threshold percentage."""
    threshold = threshold_pct * len(df)
    dropped = df.columns[df.isna().sum() > threshold].tolist()
    print(f"Dropped columns ({int(threshold_pct*100)}% threshold): {dropped}")
    return df.loc[:, df.isna().sum() <= threshold]

In [3]:
df.describe()

,cement,cement_grade,silica_fume,fly_ash,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,metakaolin,ggbfs,...,fiber2_length,fiber2_diameter,fiber2_tensile_strength,fiber2_youngs_modulus,water,sp_amount,curing_temp,curing_humidity,curing_pressure,cs_28d
count,2072.000000,1274.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,2072.000000,...,118.000000,118.000000,92.000000,58.000000,2072.000000,2064.000000,1915.000000,838.000000,71.000000,2072.000000
mean,786.164049,48.925432,180.836110,59.545347,13.614966,58.928427,32.377780,2.341776,4.903378,42.046187,...,17.528390,0.124839,2711.413043,111.379310,195.271364,31.865909,56.187833,91.202148,1.839437,150.200085
std,202.424895,4.833798,90.045347,143.872803,62.890751,113.392827,123.678095,22.752543,31.737475,141.964661,...,8.708424,0.106801,625.629497,48.480355,43.374299,17.624763,47.580591,12.183882,0.302125,36.372613
min,170.000000,42.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.050000,0.000000,400.000000,88.000000,110.000000,2.640000,20.000000,50.000000,1.200000,80.000000
25%,664.000000,42.500000,133.425000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,12.000000,0.030000,2500.000000,88.000000,166.700000,17.470000,20.000000,90.000000,1.850000,124.812500
50%,788.500000,52.500000,197.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,18.000000,0.160000,3030.000000,88.000000,184.000000,30.000000,23.000000,95.000000,2.000000,145.200000
75%,900.000000,52.500000,230.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,23.750000,0.200000,3030.000000,88.000000,214.400000,44.200000,90.000000,100.000000,2.000000,170.000000
max,1856.700000,53.000000,617.650000,1152.000000,1058.200000,528.000000,1067.000000,481.060000,510.000000,768.000000,...,36.000000,0.500000,4300.000000,250.000000,355.150000,151.700000,210.000000,100.000000,2.000000,298.000000


In [4]:
# 50% missing data threshold
df_raw_50 = df.copy()
df_raw_50 = drop_by_threshold(df_raw_50, 0.5)
df_raw_50 = df_raw_50.reset_index(drop=True)
print(f"50% threshold shape: {df_raw_50.shape}")

Dropped columns (50% threshold): ['fly_ash_type', 'slag_type', 'filler_type', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
50% threshold shape: (2072, 31)


## Create Datasets with Different Missing Data Thresholds

In [5]:
# 20% missing data threshold
df_raw_20 = df.copy()
df_raw_20 = drop_by_threshold(df_raw_20, 0.2)
df_raw_20 = df_raw_20.reset_index(drop=True)
print(f"20% threshold shape: {df_raw_20.shape}")

Dropped columns (20% threshold): ['cement_grade', 'fly_ash_type', 'slag_type', 'filler_type', 'fiber1_type', 'fiber1_amount', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_type', 'fiber2_amount', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
20% threshold shape: (2072, 26)


In [6]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = [
    ['fly_ash', 'fly_ash_type'],
    ['slag', 'slag_type'],
    ['filler', 'filler_type'],
    ['sand', 'sand_type'],
    ['fiber1_amount', 'fiber1_type'],
    ['fiber2_amount', 'fiber2_type'],
    ['sp_amount', 'sp_type']
]

In [7]:
def recode_semantic_missingness(df, amount_col, type_col):
    """Recode missing values based on material amount.
    
    Rules:
    - amount > 0 & type NaN → 'unknown_type' (material used, type unknown)
    - amount NaN/0 & type NaN → 'not_applicable' (material not used)
    """
    # Material used but type unknown
    mask1 = (df[amount_col] > 0) & df[type_col].isna()
    df.loc[mask1, type_col] = 'unknown_type'

    # Material not used (amount 0 or NaN) and type NaN
    mask2 = (df[amount_col].isna() | (df[amount_col] == 0)) & df[type_col].isna()
    df.loc[mask2, type_col] = 'not_applicable'
    df.loc[mask2, amount_col] = 0

    return df

# Apply semantic recoding
df_semantic_recoded = df.copy()

for amount_col, type_col in amount_type_twins:
    before_amount = df[amount_col].isna().sum()
    before_type = df[type_col].isna().sum()
    
    recode_semantic_missingness(df_semantic_recoded, amount_col, type_col)
    
    print(f"\n{amount_col} / {type_col}")
    print(f"  NaN amount:      {before_amount} → {df[amount_col].isna().sum()}  (true reporting gap)")
    print(f"  NaN type:        {before_type} → {df[type_col].isna().sum()}  (true reporting gap)")
    print(f"  unknown_type:    {(df[type_col] == 'unknown_type').sum()}  (recoded)")
    print(f"  not_applicable:  {(df[type_col] == 'not_applicable').sum()}  (recoded)")


fly_ash / fly_ash_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1648 → 1648  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

slag / slag_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1980 → 1980  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

filler / filler_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1712 → 1712  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

sand / sand_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        81 → 81  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber1_amount / fiber1_type
  NaN amount:      669 → 669  (true reporting gap)
  NaN type:        669 → 669  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber2_amount / fiber2_type
  NaN amount:      1954 → 1954  (true repo

In [8]:
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")

Dropped columns (50% threshold): ['fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 50% threshold shape: (2072, 36)


## Create Final Datasets (Recoded + Threshold)

In [9]:
# Recoded + 20% missing data threshold
df_semantic_recod_20 = df_semantic_recoded.copy()
df_semantic_recod_20 = drop_by_threshold(df_semantic_recod_20, 0.2)
print(f"Recoded + 20% threshold shape: {df_semantic_recod_20.shape}")


Dropped columns (20% threshold): ['cement_grade', 'fiber1_length', 'fiber1_diameter', 'fiber1_tensile_strength', 'fiber1_youngs_modulus', 'fiber2_length', 'fiber2_diameter', 'fiber2_tensile_strength', 'fiber2_youngs_modulus', 'curing_humidity', 'curing_pressure']
Recoded + 20% threshold shape: (2072, 33)


In [10]:
df_semantic_recod_50.to_csv("Datasets/processed//uhpc_dataset/semantic_recoding_features_50.csv", index=False)
df_semantic_recod_20.to_csv("Datasets/processed//uhpc_dataset/semantic_recoding_features_20.csv", index=False)
df_raw_50.to_csv("Datasets/processed//uhpc_dataset/raw_dropped_features_50.csv", index=False)
df_raw_20.to_csv("Datasets/processed//uhpc_dataset/raw_dropped_features_20.csv", index=False)

In [11]:
df_semantic_recod_20.isnull().sum()[df_semantic_recod_20.isnull().sum() > 0]

sand_max_size    276
sp_amount          8
curing_temp      157
dtype: int64

In [12]:
df_semantic_recod_20['sand'][df_semantic_recod_20['sand_max_size'].isna()].describe()

count     276.000000
mean      816.853913
std       397.325703
min         0.000000
25%       653.000000
50%       973.350000
75%      1075.000000
max      1756.480000
Name: sand, dtype: float64

In [13]:
num_cols = df_semantic_recod_50.select_dtypes(include='number').columns

Q1 = df_semantic_recod_50[num_cols].quantile(0.25)
Q3 = df_semantic_recod_50[num_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_counts = ((df_semantic_recod_50[num_cols] < (Q1 - 1.5 * IQR)) | (df_semantic_recod_50[num_cols] > (Q3 + 1.5 * IQR))).sum()
outlier_pct = (outlier_counts / len(df_semantic_recod_50) * 100).round(1)

summary = pd.DataFrame({'outlier_count': outlier_counts, 'outlier_%': outlier_pct})
print(summary[summary['outlier_count'] > 0].sort_values('outlier_%', ascending=False))

                  outlier_count  outlier_%
fiber1_length               523       25.2
quartz_powder               514       24.8
fly_ash                     435       21.0
filler                      360       17.4
ggbfs                       235       11.3
fiber1_diameter             215       10.4
sand                        190        9.2
glass_powder                189        9.1
water                       129        6.2
limestone_powder            126        6.1
fiber2_amount               118        5.7
slag                         96        4.6
nano_sio2                    95        4.6
curing_temp                  76        3.7
sand_max_size                77        3.7
metakaolin                   73        3.5
cement                       53        2.6
cs_28d                       46        2.2
nano_caco3                   44        2.1
rice_husk_ash                28        1.4
sp_amount                    27        1.3
fiber1_amount                21        1.0
silica_fume

In [14]:
Q1 = df_semantic_recod_50['cs_28d'].quantile(0.25)
Q3 = df_semantic_recod_50['cs_28d'].quantile(0.75)
IQR = Q3 - Q1
mask = (df_semantic_recod_50['cs_28d'] < Q1 - 1.5*IQR) | (df_semantic_recod_50['cs_28d'] > Q3 + 1.5*IQR)
print(df_semantic_recod_50.loc[mask, 'cs_28d'].describe())

count     46.000000
mean     258.869565
std       14.035841
min      239.000000
25%      248.250000
50%      257.000000
75%      268.000000
max      298.000000
Name: cs_28d, dtype: float64


In [15]:
df_semantic_recod_50.select_dtypes(include='object').shape[1]

C:\Users\jasur\AppData\Local\Temp\ipykernel_6304\2735213181.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_semantic_recod_50.select_dtypes(include='object').shape[1]


10